### Подбор гиперпараметров

In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

In [2]:
import warnings
for warn in [UserWarning, FutureWarning]: warnings.filterwarnings("ignore", category = warn)

In [3]:
data = pd.read_csv('free_text_EmoSurv.csv')

In [4]:
users = data['userid'].unique().tolist()
train_users, test_users = train_test_split(users, test_size = 0.2, random_state = 0) # делим на train и test по пользователям

train_data = data[data['userid'].isin(train_users)].copy()

Дополнительно выделим 10% в качестве валидационной выборки

In [5]:
test_users, valid_users = train_test_split(test_users, test_size = 0.5, random_state = 0)

test_data = data[data['userid'].isin(test_users)].copy()
valid_data = data[data['userid'].isin(valid_users)].copy()

In [7]:
feature_cols = ['D1U1', 'D1U2', 'D1D2', 'U1D2', 'U1U2', 'D1U3', 'D1D3', 'is_punctuation', 'is_backspace', 'is_enter', 'hour']

scaler_X = StandardScaler()
train_data[feature_cols] = scaler_X.fit_transform(train_data[feature_cols])
test_data[feature_cols] = scaler_X.transform(test_data[feature_cols])
valid_data[feature_cols] = scaler_X.transform(valid_data[feature_cols])

In [8]:
def create_sequences(data, seq_length, step):
    X, y = [], []

    for id in data['userid'].unique():
        user_data = data[data['userid'] == id].sort_values('index')        
        user_X = user_data.drop(columns=['emotionIndex', 'index', 'userid', 'Unnamed: 0']).values
        user_y = user_data['emotionIndex'].values

        if len(user_data) < seq_length + 1:
            continue

        for j in range(0, len(user_data) - seq_length, step):
            X.append(user_X[j:j+seq_length])
            y.append(user_y[j+seq_length])

    X = np.array(X, dtype = np.float32)
    y = np.array(y, dtype=np.int64)
    
    return torch.FloatTensor(X), torch.LongTensor(y)

In [9]:
SEQ_LEN = 32  # длина последовательности
STEP = 16     # шаг для последовательности
X_train, y_train = create_sequences(train_data, SEQ_LEN, STEP)
X_test, y_test = create_sequences(test_data, SEQ_LEN, STEP)
X_valid, y_valid = create_sequences(valid_data, SEQ_LEN, STEP)

In [10]:
train_dataset = TensorDataset(X_train, y_train)
valid_dataset = TensorDataset(X_valid, y_valid)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)  
valid_loader = DataLoader(valid_dataset, batch_size = 16, shuffle = False) 
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)    

In [11]:
class DetectorAttentionGRU(nn.Module):
    def __init__(self, input_size = 1, hidden_size = 32, num_layers = 2, num_classes = 1, dropout = 0.1, bidirectional = False):
        super(DetectorAttentionGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)

        self.gru = nn.GRU(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout,
            bidirectional = bidirectional
        )

        self.output_size = hidden_size * (2 if bidirectional else 1)

        self.attention = nn.Sequential(
            nn.Linear(self.output_size, 1)
        )

        self.fc = nn.Linear(self.output_size, num_classes) 

        
    def forward(self, x):
        out, h_n = self.gru(x)

        attn = self.attention(out)
        attention_weights = F.softmax(attn, dim = 1)

        context = torch.sum(attention_weights * out, dim = 1)  # [batch_size, output_size]
        context = self.dropout(context)
        output = self.fc(context)

        return output

In [12]:
# вычисляем веса для классов (обратно пропорционально частоте)
class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2, 3, 4]),
    y=y_train.numpy()
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [13]:
def validate(model, val_loader):
    model.eval()  # режим оценки (выключаем dropout, batch norm)
    all_preds = []
    all_targets = []
    
    with torch.no_grad(): 
        for X_batch, y_batch in val_loader:
            output = model(X_batch)  
            preds = torch.argmax(output, dim = 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())
    
    f1 = f1_score(all_targets, all_preds, average='macro')
    return f1

In [14]:
def objective(trial):
    learning_rate = trial.suggest_float('learning rate', 0, 0.1)
    hidden_size = trial.suggest_int('hidden_size', 64, 128, step = 16)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)

    model = DetectorAttentionGRU(input_size = 11, hidden_size = hidden_size, 
                                 num_layers = 2, num_classes = 5, dropout = dropout, bidirectional = True)
    optimizer = optim.Adam(model.parameters(), lr = learning_rate)

    # обучаем 
    for epoch in range(20):
        model.train()
        for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                output = model(X_batch)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()
        
    # валидируем
    res = validate(model, valid_loader)
    return res


study = optuna.create_study(direction='maximize', study_name='GRU_hyperparametrs')
optuna.logging.disable_default_handler()
study.optimize(objective, n_trials = 30)

trial = study.best_trial

print("Best F1 score: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2026-06-25 21:47:51,419] A new study created in memory with name: GRU_hyperparametrs


Best F1 score: 0.35764254613311214
Best hyperparameters: {'learning rate': 0.01126473668777453, 'hidden_size': 80, 'dropout': 0.2951056865914082}


In [ ]:
def model_analysis(model, total_epochs, model_optimizer, criterion):
    train_f1 = []
    train_loss = []

    for epoch in range(total_epochs):
        model.train()
        total_loss = 0
        targets, predictions = [], []
        
        for X_batch, y_batch in train_loader:
            model_optimizer.zero_grad()
            output = model(X_batch)
            
            loss = criterion(output, y_batch)
            loss.backward()
            model_optimizer.step()

            preds = torch.argmax(output, dim = 1)
            total_loss += loss.item() * X_batch.size(0)
            targets.extend(y_batch.cpu().numpy())
            predictions.extend(preds.cpu().numpy())
        
        avg_loss = total_loss / len(train_loader.dataset)
        train_loss.append(avg_loss)
        f1 = f1_score(targets, predictions, average='macro')
        train_f1.append(f1)
        
        if epoch % 10 == 9 or epoch == 0:
            print(f'Epoch {epoch+1}/{total_epochs}, Loss: {avg_loss:.6f}')
        
    return train_loss, train_f1

In [ ]:
def plot_train_dynamics(model, train_loss, train_f1, title):
    fig, axs = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(title, fontweight='bold')

    axs[0].plot(train_loss, color='royalblue')
    axs[0].set_xlabel('epoch')
    axs[0].set_ylabel('loss')
    axs[0].set_title('Loss dynamics')


    all_weights = []
    for param in model.parameters():
        if param.requires_grad:
            all_weights.extend(param.detach().numpy().flatten())

    axs[1].plot(train_f1, color='teal')
    axs[1].set_title('F1 score dynamics')
    axs[1].set_xlabel('epoch')
    axs[1].set_ylabel('f1_score')

    plt.show()

In [ ]:
attn_model_bidirect = DetectorAttentionGRU(input_size = 11, hidden_size = 64, num_layers = 2, num_classes = 5, dropout = 0.2, bidirectional = True)
optimizer = optim.Adam(attn_model_bidirect.parameters(), lr = 0.001)

train_loss_attn_b, train_f1_attn_b = model_analysis(attn_model_bidirect, NUM_EPOCHS, optimizer, criterion)